# Medical Imaging Clinical AI: End-to-End Demo

## Project Question

**Can combining medical images with structured clinical metadata improve disease-risk prediction compared with image-only AI?**

This notebook uses **MedMNIST** as a public medical imaging dataset and demonstrates:

1. Load image dataset  
2. Load/create metadata CSV  
3. Train CNN baseline  
4. Train multimodal image + metadata model  
5. Compare metrics  
6. Show confusion matrix and classification report  
7. Show ROC curve and precision-recall curve  
8. Show Grad-CAM explanation  
9. Provide clinical interpretation  


In [ ]:
# Optional dependency installation
# Run this cell only if an import fails.

import sys
from pathlib import Path

def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data_utils.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing src/data_utils.py."
    )

PROJECT_ROOT = locate_project_root(Path.cwd())
REQUIREMENTS = PROJECT_ROOT / "requirements.txt"
print("Project root:", PROJECT_ROOT)
print("Requirements file:", REQUIREMENTS)

# Uncomment when packages need to be installed into this notebook kernel:
# !{sys.executable} -m pip install -r "{REQUIREMENTS}"


In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)

def locate_project_root(start: Path) -> Path:
    """Find the nearest parent containing the expected src package."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        required = (
            candidate / "src" / "__init__.py",
            candidate / "src" / "data_utils.py",
            candidate / "src" / "models.py",
            candidate / "src" / "train_eval.py",
        )
        if all(path.is_file() for path in required):
            return candidate
    raise FileNotFoundError(
        f"Could not locate the project root from {start}. "
        "Keep the notebook inside the project or launch Jupyter from the project root."
    )

ROOT = locate_project_root(Path.cwd())
root_string = str(ROOT)
if root_string in sys.path:
    sys.path.remove(root_string)
sys.path.insert(0, root_string)

# Clear a cached unrelated package named src after an earlier failed run.
loaded_src = sys.modules.get("src")
if loaded_src is not None:
    loaded_file = Path(getattr(loaded_src, "__file__", "") or "").resolve()
    expected_src = (ROOT / "src").resolve()
    if expected_src not in loaded_file.parents:
        for module_name in list(sys.modules):
            if module_name == "src" or module_name.startswith("src."):
                del sys.modules[module_name]

from src.data_utils import (
    create_synthetic_metadata,
    normalize_metadata,
    ImageOnlyDataset,
    MultimodalDataset,
)
from src.models import GradCAM, MultimodalClinicalModel, SimpleCNN
from src.train_eval import (
    evaluate_image_model,
    evaluate_multimodal_model,
    train_image_model,
    train_multimodal_model,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FIGURE_DIR = ROOT / "outputs" / "figures"
TABLE_DIR = ROOT / "outputs" / "tables"
DATA_DIR = ROOT / "data"
for directory in (FIGURE_DIR, TABLE_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Notebook directory:", Path.cwd().resolve())
print("Project root:", ROOT)
print("src package:", __import__("src").__file__)
print("Device:", device)


## 1. Load public medical imaging dataset: MedMNIST

This demo uses `PneumoniaMNIST`, a binary chest X-ray dataset from MedMNIST.  
It is small enough to run locally and appropriate for portfolio demonstration.


In [ ]:
from medmnist import PneumoniaMNIST

train_data = PneumoniaMNIST(split="train", download=True)
test_data = PneumoniaMNIST(split="test", download=True)

X_train = train_data.imgs
y_train = train_data.labels.reshape(-1)
X_test = test_data.imgs
y_test = test_data.labels.reshape(-1)

print("Train images:", X_train.shape)
print("Test images:", X_test.shape)
print("Train label distribution:", np.bincount(y_train))
print("Test label distribution:", np.bincount(y_test))

In [ ]:
plt.figure(figsize=(8, 3))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"y={y_train[i]}")
    plt.axis("off")
plt.suptitle("Sample MedMNIST Images")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sample_medmnist_images.png", bbox_inches="tight")
plt.show()

## 2. Load / create metadata CSV

MedMNIST provides image-label data but not full structured clinical metadata.  
For this portfolio version, we create a synthetic metadata CSV with:

- age  
- sex  
- prior condition  
- scanner site  

This shows the multimodal workflow. In a real clinical project, replace this CSV with real structured metadata.


In [ ]:
train_meta_path = DATA_DIR / "train_metadata.csv"
test_meta_path = DATA_DIR / "test_metadata.csv"

train_meta = create_synthetic_metadata(y_train, train_meta_path)
test_meta = create_synthetic_metadata(y_test, test_meta_path)

display(train_meta.head())

# Fit normalization statistics on training metadata only, then reuse them.
X_meta_train, metadata_normalizer = normalize_metadata(
    train_meta,
    return_normalizer=True,
)
X_meta_test = normalize_metadata(
    test_meta,
    normalizer=metadata_normalizer,
)

print("Training metadata shape:", X_meta_train.shape)
print("Test metadata shape:", X_meta_test.shape)
print("Training normalizer:", metadata_normalizer)


## 3. Create dataloaders

In [ ]:
train_image_ds = ImageOnlyDataset(X_train, y_train)
test_image_ds = ImageOnlyDataset(X_test, y_test)

train_multi_ds = MultimodalDataset(X_train, y_train, X_meta_train)
test_multi_ds = MultimodalDataset(X_test, y_test, X_meta_test)

train_image_loader = DataLoader(train_image_ds, batch_size=64, shuffle=True)
test_image_loader = DataLoader(test_image_ds, batch_size=128, shuffle=False)

train_multi_loader = DataLoader(train_multi_ds, batch_size=64, shuffle=True)
test_multi_loader = DataLoader(test_multi_ds, batch_size=128, shuffle=False)

print("Dataloaders ready")

## 4. Train CNN baseline: image-only model

In [ ]:
cnn_model = SimpleCNN(num_classes=2)

cnn_history = train_image_model(
    cnn_model,
    train_image_loader,
    epochs=2,
    lr=1e-3,
    device=device,
)

## 5. Train multimodal model: image + metadata

In [ ]:
multi_model = MultimodalClinicalModel(metadata_dim=X_meta_train.shape[1], num_classes=2)

multi_history = train_multimodal_model(
    multi_model,
    train_multi_loader,
    epochs=2,
    lr=1e-3,
    device=device,
)

## 6. Evaluate and compare models

In [ ]:
cnn_metrics, y_true, cnn_pred, cnn_prob = evaluate_image_model(
    cnn_model,
    test_image_loader,
    device=device,
)

multi_metrics, y_true_multi, multi_pred, multi_prob = evaluate_multimodal_model(
    multi_model,
    test_multi_loader,
    device=device,
)

if not np.array_equal(y_true, y_true_multi):
    raise RuntimeError("Image-only and multimodal evaluation labels do not match.")

# The graph-enhanced row is a conceptual extension, not a trained GNN result.
eval_table = pd.DataFrame(
    [
        {
            "Model": "CNN baseline",
            "Input": "Image only",
            "Accuracy": cnn_metrics["accuracy"],
            "AUC": cnn_metrics["roc_auc"],
            "F1": cnn_metrics["f1"],
            "Sensitivity": cnn_metrics["sensitivity_recall"],
            "Specificity": cnn_metrics["specificity"],
            "Clinical meaning": "Imaging-only benchmark",
        },
        {
            "Model": "Multimodal",
            "Input": "Image + metadata",
            "Accuracy": multi_metrics["accuracy"],
            "AUC": multi_metrics["roc_auc"],
            "F1": multi_metrics["f1"],
            "Sensitivity": multi_metrics["sensitivity_recall"],
            "Specificity": multi_metrics["specificity"],
            "Clinical meaning": "Closer to a clinical workflow",
        },
        {
            "Model": "Graph-enhanced",
            "Input": "Similar cases",
            "Accuracy": np.nan,
            "AUC": np.nan,
            "F1": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "Clinical meaning": "Conceptual case-based reasoning extension",
        },
    ]
)

display(eval_table)
eval_table.to_csv(TABLE_DIR / "model_comparison.csv", index=False)


## 7. Confusion matrix and classification report

In [ ]:
cm = confusion_matrix(y_true_multi, multi_pred)

disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Pneumonia"])
disp.plot()
plt.title("Confusion Matrix: Multimodal Model")
plt.savefig(FIGURE_DIR / "confusion_matrix_multimodal.png", bbox_inches="tight")
plt.show()

report = classification_report(
    y_true_multi,
    multi_pred,
    target_names=["Normal", "Pneumonia"],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_csv(TABLE_DIR / "classification_report_multimodal.csv")

## 8. ROC curve and precision-recall curve

In [ ]:
RocCurveDisplay.from_predictions(y_true, cnn_prob, name="CNN baseline")
RocCurveDisplay.from_predictions(y_true_multi, multi_prob, name="Multimodal")
plt.title("ROC Curve Comparison")
plt.savefig(FIGURE_DIR / "roc_curve_comparison.png", bbox_inches="tight")
plt.show()

PrecisionRecallDisplay.from_predictions(y_true, cnn_prob, name="CNN baseline")
PrecisionRecallDisplay.from_predictions(y_true_multi, multi_prob, name="Multimodal")
plt.title("Precision-Recall Curve Comparison")
plt.savefig(FIGURE_DIR / "precision_recall_curve_comparison.png", bbox_inches="tight")
plt.show()

## 9. Model comparison bar chart

In [ ]:
plot_df = eval_table.dropna(subset=["Accuracy"]).copy()

x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(9, 5))
plt.bar(x - width, plot_df["Accuracy"], width, label="Accuracy")
plt.bar(x, plot_df["AUC"], width, label="AUC")
plt.bar(x + width, plot_df["F1"], width, label="F1")
plt.xticks(x, plot_df["Model"])
plt.ylim(0, 1)
plt.title("Model Comparison")
plt.ylabel("Score")
plt.legend()
plt.grid(axis="y")
plt.savefig(FIGURE_DIR / "model_comparison_bar_chart.png", bbox_inches="tight")
plt.show()

## 10. Grad-CAM explanation

Grad-CAM highlights image regions that influenced the CNN model prediction.  
This improves interpretability but is not a substitute for clinician review.


In [ ]:
cnn_model.to(device)
target_layer = cnn_model.conv3
gradcam = GradCAM(cnn_model, target_layer)

try:
    sample_image, sample_label = test_image_ds[0]
    sample_input = sample_image.unsqueeze(0).to(device)
    cam = gradcam.generate(sample_input)

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(sample_image.squeeze().cpu().numpy(), cmap="gray")
    plt.title(f"Original image | label={sample_label.item()}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(sample_image.squeeze().cpu().numpy(), cmap="gray")
    plt.imshow(cam.cpu().numpy(), alpha=0.45)
    plt.title("Grad-CAM heatmap")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "gradcam_heatmap.png",
        bbox_inches="tight",
        dpi=160,
    )
    plt.show()
finally:
    gradcam.close()


## 11. Final clinical interpretation

### Answer to the project question

This notebook tests whether combining image features with structured metadata improves disease-risk prediction compared with an image-only CNN baseline.

### Real evaluation section

The results are saved to:

```text
outputs/tables/real_evaluation_table.csv
```

### Clinical meaning

- **CNN baseline**: image-only benchmark.
- **Multimodal model**: closer to real clinical workflow because clinicians often use image findings plus patient context.
- **Graph-enhanced model**: conceptual extension using similar historical cases.

### Important limitation

This is a portfolio-level AI demonstration. It is decision support, not diagnosis.  
The metadata here is synthetic because MedMNIST does not include full structured clinical metadata.

For real use, replace the synthetic metadata with validated clinical variables and perform external validation, calibration, fairness checks, and clinician review.
